In [ ]:
import cv2
import numpy as np

video_path = "target.mov"
background_path = "background.png"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()
if not ret:
    raise ValueError("Cannot read video")

h, w = frame.shape[:2]

background = cv2.resize(cv2.imread(background_path), (w, h))

out = cv2.VideoWriter(
    "output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    30,
    (w, h)
)

# =========================
# Target color ranges
# =========================
lower1 = np.array([0, 160, 40])
upper1 = np.array([10, 255, 255])

lower2 = np.array([165, 160, 40])
upper2 = np.array([179, 255, 255])

# =========================
# Temporal memory
# =========================
mask_memory = np.zeros((h, w), dtype=np.float32)
alpha_t = 0.5

# predefine kernels (avoid recreating every frame)
k3 = np.ones((3,3), np.uint8)
k5 = np.ones((5,5), np.uint8)
k9 = np.ones((9,9), np.uint8)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # =========================
    # Motion mask
    # =========================
    diff = cv2.absdiff(frame, background)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    motion_mask = (gray > 20).astype(np.float32)
    motion_mask = cv2.GaussianBlur(motion_mask, (7,7), 0)

    # =========================
    # HSV
    # =========================
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # =========================
    # Color mask
    # =========================
    mask1 = cv2.inRange(hsv, lower1, upper1)
    mask2 = cv2.inRange(hsv, lower2, upper2)
    color_mask = ((mask1 | mask2) > 0).astype(np.uint8)

    color_mask = cv2.medianBlur(color_mask * 255, 5)
    color_mask = (color_mask > 0).astype(np.uint8)

    # =========================
    # Skin protection
    # =========================
    skin_mask = cv2.inRange(hsv, (0,20,70), (35,255,255))
    skin_mask = cv2.morphologyEx(skin_mask, cv2.MORPH_CLOSE, k9)
    skin_mask = cv2.GaussianBlur(skin_mask, (7,7), 0)
    skin_mask = (skin_mask > 50).astype(np.uint8)

    # =========================
    # Raw mask
    # =========================
    mask = color_mask.astype(np.float32)

    mask *= (1 + 0.5 * motion_mask)
    mask = np.clip(mask, 0.0, 1.0)

    mask *= (1 - skin_mask.astype(np.float32))

    # =========================
    # Clean + edge recovery
    # =========================
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k5)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, k3)
    mask = cv2.dilate(mask, k3, 1)

    # =========================
    # Temporal memory
    # =========================
    mask_memory = (1 - alpha_t) * mask_memory + alpha_t * mask

    # =========================
    # Core + alpha
    # =========================
    core = (mask_memory > 0.45).astype(np.float32)
    alpha = np.maximum(mask_memory, core)

    # =========================
    # Edge feathering
    # =========================
    mask_bin = (mask_memory > 0.3).astype(np.uint8)

    edge = cv2.morphologyEx(mask_bin, cv2.MORPH_GRADIENT, k3)
    feather = cv2.dilate(edge, k5, 1)

    feather = cv2.GaussianBlur(feather.astype(np.float32), (15,15), 0)
    feather = np.clip(feather, 0, 1)

    alpha = np.maximum(alpha, feather * 0.8)
    alpha = np.clip(alpha, 0.0, 1.0)

    # =========================
    # Blend
    # =========================
    alpha_3 = cv2.merge([alpha, alpha, alpha])
    result = (frame * (1 - alpha_3) + background * alpha_3).astype(np.uint8)

    out.write(result)
    cv2.imshow("result", result)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

print("Done")